In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [1]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import SimpleDirectoryReader

# Load environment variables
_ = load_dotenv(find_dotenv())



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/promise/projects/RAG_PMG/.venv/lib/python3.11/si
[nltk_data]     te-packages/llama_index/core/_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
from llama_index.core.node_parser import SentenceSplitter

documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()
node_parser = SentenceSplitter(chunk_size=2048)
nodes = node_parser.get_nodes_from_documents(documents)
    


In [3]:
nodes[1].text

'Preach My Gospel (D&C 50:14) (Page 5)\n\n| Chapters | First Presidency Message | page no |\n| --- | --- | --- |\n|  | Introduction: How Can I Best Use Preach My Gospel? | vii |\n| 1 | What Is My Purpose as a Missionary? | 1 |\n| 2 | How Do I Study Effectively and Prepare to Teach? | 17 |\n| 3 | What Do I Study and Teach? | 29 |\n|  | Lesson 1: The Message of the Restoration of the Gospel of Jesus Christ | 31 |\n|  | Lesson 2: The Plan of Salvation | 47 |\n|  | Lesson 3: The Gospel of Jesus Christ | 60 |\n|  | Lesson 4: The Commandments</l1? | 71 |\n|  | Lesson 5: Laws and Ordinances | 82 |\n| 4 | How Do I Recognize and Understand the Spirit? | 89 |\n| 5 | What Is the Role of the Book of Mormon? | 103 |\n| 6 | How Do I Develop Christlike Attributes? | 115 |\n| 7 | How Can I Better Learn My Mission Language? | 127 |\n| 8 | How Do I Use Time Wisely? | 137 |\n| 9 | How Do I Find People to Teach? | 155 |\n| 10 | How Can I Improve My Teaching Skills? | 175 |\n| 11 | How Do I Help People Mak

In [4]:
print(len(nodes))

509


## with ragas

In [5]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from llama_index.llms.openai import OpenAI

from llama_index.embeddings.openai import OpenAIEmbedding


# Initialize the LLMs with LlamaIndex
generator_llm = OpenAI(model="gpt-4o-mini") 
critic_llm = OpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbedding()

# Create the TestsetGenerator using LlamaIndex
generator = TestsetGenerator.from_llama_index(
    generator_llm,
    critic_llm,
    embeddings
)

# Change resulting question type distribution
distributions = {
    simple: 0.33,
    multi_context: 0.33,
    reasoning: 0.34
}



/home/promise/projects/RAG_PMG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
batch_size = 100
total_nodes = len(nodes)

# Split the nodes into batches of 100, but for the last two batches, combine them
batches = [nodes[i:i + batch_size] for i in range(0, total_nodes - 109, batch_size)]  # First 4 batches (100 nodes each)

# Combine the last 109 nodes into one batch
last_batch = nodes[400:509]  # Combining nodes[400:509]

# Append the last combined batch to the list of batches
batches.append(last_batch)
len(batches)


5

In [9]:

# Now you have 5 batches, each in `batches`
batch_1 = batches[0]  # Nodes 0-99
batch_2 = batches[1]  # Nodes 100-199
batch_3 = batches[2]  # Nodes 200-299
batch_4 = batches[3]  # Nodes 300-399
batch_5 = batches[4]  # Nodes 400-509 (last combined batch of 109 nodes)


In [ ]:
batches = [batch_1, batch_2, batch_3, batch_4, batch_5]  # Assuming you have all the batches in a list

for idx, batch in enumerate(batches, 1):
    print(f"generating batches for batch {idx}")
    print()
    testset = generator.generate_with_llamaindex_docs(batch, 50, distributions)
    batch_df = testset.to_pandas()
    batch_df.reset_index(drop=True, inplace=True)
    batch_df.to_csv(f"../dataset_eval/batch_{idx}.csv", index=False)


In [23]:
import pandas as pd
import os

def load_dataframe(file):
    return pd.read_csv(file)

def merge_dataframe(main_df, batch_df):
    if main_df is None:
        main_df = batch_df

    else:
        main_df = pd.concat([main_df, batch_df], ignore_index=True)
    return main_df
        

In [25]:
input_dir = "../dataset_eval"
files = os.listdir(input_dir)

# Remove '10_dataset.csv' from the list
files = [file for file in files if file != '10_dataset.csv']

# load and merge dataframe
result = None
for file in files:
    input_file = os.path.join(input_dir, file)
    load_file = load_dataframe(input_file)
    
    result = merge_dataframe(result, load_file)
    
# Save the final merged DataFrame to a new CSV file
if result is not None:
    result.to_csv("../dataset_eval/merge_batch.csv", index=False)
else:
    print("No data was loaded from the CSV files.")    
    




In [28]:
dataset = load_dataframe("../dataset_eval/merge_batch.csv")
dataset.tail()

,question,contexts,ground_truth,evolution_type,metadata,episode_done
234,What links exist between the Holy Spirit's act...,"[""Personal Study (Page 114)\n- Record your spi...",The context mentions that the book of Acts has...,reasoning,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
235,What helps people resist temptation during rep...,['Repentance and Addiction Recovery (Page 201)...,"The gift of the Holy Ghost, received through b...",reasoning,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
236,What does a Ward Mission Leader do to coordina...,['Ward Mission Leader (Page 232)\nWard Mission...,A Ward Mission Leader coordinates the work of ...,reasoning,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
237,"How do the scriptures, like the Book of Mormon...",['Remember This (Page 206)\n- As people are ta...,"The scriptures, especially the Book of Mormon,...",reasoning,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
238,What steps ensure candidates are ready for the...,['Consider This (Page 217)\n- What do I need t...,To ensure candidates are ready for the baptism...,multi_context,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True


In [29]:
# Remove duplicates based on just the 'Question' column
df_cleaned = dataset.drop_duplicates(subset=['question'])
df_cleaned.head()

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What strategies can be used to make a message ...,['Teach with Clarity (Page 44)\n\nAt the end o...,The context suggests that to make a message ea...,simple,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
1,What should teachers do with unfamiliar words ...,['Teach with Clarity (Page 44)\n\nAt the end o...,Teachers should learn how to define unfamiliar...,simple,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
2,What are some effective study techniques to en...,"['Search, Ponder, and Remember (Page 36)\n- Be...",Effective study techniques to enhance understa...,simple,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
3,What is the purpose of using a study journal i...,['Use Study Resources (Page 37)\n- Use the stu...,The purpose of using a study journal in your s...,simple,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True
4,What does 3 Nephi 11:31–41 reveal about the go...,['Scripture Study (Page 20)\nWhat is the gospe...,The answer to given question is not present in...,simple,"[{'file_path': '../data/Pmg_lds.md', 'file_nam...",True


In [30]:
len(df_cleaned)

230

In [31]:
# Select only the 'question' and 'ground_truth' columns
df_selected = df_cleaned[['question', 'contexts', 'ground_truth']]

# Display the selected columns
df_selected.head()


,question,contexts,ground_truth
0,What strategies can be used to make a message ...,['Teach with Clarity (Page 44)\n\nAt the end o...,The context suggests that to make a message ea...
1,What should teachers do with unfamiliar words ...,['Teach with Clarity (Page 44)\n\nAt the end o...,Teachers should learn how to define unfamiliar...
2,What are some effective study techniques to en...,"['Search, Ponder, and Remember (Page 36)\n- Be...",Effective study techniques to enhance understa...
3,What is the purpose of using a study journal i...,['Use Study Resources (Page 37)\n- Use the stu...,The purpose of using a study journal in your s...
4,What does 3 Nephi 11:31–41 reveal about the go...,['Scripture Study (Page 20)\nWhat is the gospe...,The answer to given question is not present in...


In [32]:
# Filter out rows where 'ground_truth' contains the unwanted phrase
df_filtered = df_selected[~df_selected['ground_truth'].str.contains("The answer to given question is not present in| As an AI|The context does not provide|I'm sorry,|The text doesn't provide information")]

# Display the filtered DataFrame
df_filtered.head()


,question,contexts,ground_truth
0,What strategies can be used to make a message ...,['Teach with Clarity (Page 44)\n\nAt the end o...,The context suggests that to make a message ea...
1,What should teachers do with unfamiliar words ...,['Teach with Clarity (Page 44)\n\nAt the end o...,Teachers should learn how to define unfamiliar...
2,What are some effective study techniques to en...,"['Search, Ponder, and Remember (Page 36)\n- Be...",Effective study techniques to enhance understa...
3,What is the purpose of using a study journal i...,['Use Study Resources (Page 37)\n- Use the stu...,The purpose of using a study journal in your s...
6,What is the importance of organizing and summa...,['Studying and Preparing to Teach the Lessons ...,Organizing and summarizing lesson plans is imp...


In [33]:
len(df_filtered)

188

In [34]:
df_filtered.to_csv("../dataset_eval/20_dataset.csv")

## with llama index/gpt-40-mini

In [ ]:
# from llama_index.core.llama_dataset.generator import RagDatasetGenerator
# from llama_index.llms.openai import OpenAI
# import nest_asyncio

# nest_asyncio.apply()



# llm = OpenAI(model="gpt-4o-mini")

# dataset_generator = RagDatasetGenerator.from_documents(
#     documents=nodes,
#     llm=llm,
#     num_questions_per_chunk=num_questions_per_node
# )


# rag_dataset = dataset_generator.generate_dataset_from_nodes()


In [ ]:
# import pandas as pd
# df = pd.DataFrame(rag_dataset)
# size= df[1][0]

# for i in range(len(size)):
#     import pandas as pd

# # Extract the data from rag_dataset
# data = []
# for i in range(len(size)):
#     data.append({
#         'query': size[i].query,
#         'reference_contexts': size[i].reference_contexts,  # Keep contexts as a list
#         'reference_answer': size[i].reference_answer,
#         'query_by': size[i].query_by.model_name,  # Model that generated the query
#         'reference_answer_by': size[i].reference_answer_by.model_name  # Model that provided the answer
#     })

# # Create a pandas DataFrame
# df_4 = pd.DataFrame(data)

# # Display the DataFrame
# df_4.head()

    
    

In [ ]:
# df_selected_4 = df_4[['query', 'reference_contexts', 'reference_answer']]
# df_selected_4.tail()

In [ ]:
# # Filtering rows where 'reference_answer' contains placeholders
# filtered_df_4 = df_selected_4[~df_selected_4['reference_answer'].str.contains("As an AI|The context does not provide|I'm sorry,|The text doesn't provide information")]
# # filtered_df_4.head()

## with llama index 4o

In [12]:
# from llama_index.core.llama_dataset.generator import RagDatasetGenerator
# from llama_index.llms.openai import OpenAI
# import nest_asyncio

# nest_asyncio.apply()



# llm = OpenAI(model="gpt-4o-mini")

# dataset_generator = RagDatasetGenerator.from_documents(
#     documents=sample,
#     llm=llm,
#     num_questions_per_chunk=1,  # set the number of questions per nodes
# )

# rag_dataset = dataset_generator.generate_dataset_from_nodes()

In [ ]:
# size= df[1][0]
# for i in range(len(size)):
#     import pandas as pd

# # Extract the data from rag_dataset
# data = []
# for i in range(len(size)):
#     data.append({
#         'query': size[i].query,
#         'reference_contexts': size[i].reference_contexts,  # Keep contexts as a list
#         'reference_answer': size[i].reference_answer,
#         'query_by': size[i].query_by.model_name,  # Model that generated the query
#         'reference_answer_by': size[i].reference_answer_by.model_name  # Model that provided the answer
#     })

# # Create a pandas DataFrame
# df_orig = pd.DataFrame(data)

# # Display the DataFrame
# df_orig.head()

    
    

In [ ]:
# import pandas as pd
# df = pd.DataFrame(dataset)
# size= df[1][0]
# # Extract the data from rag_dataset
# data = []
# for i in range(len(size)):
#     data.append({
#         'query': size[i].query,
#         'reference_contexts': size[i].reference_contexts,  # Keep contexts as a list
#         'reference_answer': size[i].reference_answer,
#         'query_by': size[i].query_by.model_name,  # Model that generated the query
#         'reference_answer_by': size[i].reference_answer_by.model_name  # Model that provided the answer
#     })

# # Create a pandas DataFrame
# df_orig = pd.DataFrame(data)

# # Display the DataFrame
# df_orig.head()

    
    